# LLM, Logging, Retry, and Base Contract Test

Tests core infrastructure used across the app without needing to run the UI.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / 'src').exists()), cwd)
sys.path.insert(0, str(project_root / 'src'))
print('project_root:', project_root)

In [ ]:
from config.settings import AppConfig, LLMConfig
from core.base_agent import AgentRequest, AgentResult

cfg = AppConfig()
print('provider:', cfg.llm.provider)
print('model:', cfg.llm.primary_model)
print('api_key_set:', bool(cfg.llm.api_key))
print('fallbacks:', cfg.llm.fallback_models)
assert cfg.llm.model == cfg.llm.primary_model

req = AgentRequest(query='test query', data_products=['retention'])
res = AgentResult(agent_name='notebook_agent', success=True, summary='ok')
print('request:', req)
print('result dict:', res.to_dict())
assert res.to_dict()['agent'] == 'notebook_agent'

In [ ]:
from core.llm_factory import get_llm, get_structured_llm
from graph.intent import IntentClassification

llm = get_llm(cfg.llm, streaming=False)
structured = get_structured_llm(cfg.llm, IntentClassification)
print('llm:', type(llm))
print('structured:', type(structured))
assert llm is not None
assert structured is not None

RUN_LLM_CALL = False
if RUN_LLM_CALL:
    from langchain_core.messages import HumanMessage
    print(llm.invoke([HumanMessage(content='Reply with exactly: ok')]).content)
else:
    print('Skipped live LLM call. Set RUN_LLM_CALL = True to test provider credentials.')

In [ ]:
from core.logging_utils import setup_logger, with_retry

logger = setup_logger('notebook.core_test')
logger.info('notebook logging smoke test')
print('logger handlers:', len(logger.handlers))
print('propagate:', logger.propagate)
assert logger.handlers
assert logger.propagate is False

calls = {'count': 0}

@with_retry(max_retries=3, delay_seconds=0.01, backoff=1.0)
def flaky():
    calls['count'] += 1
    if calls['count'] < 2:
        raise ValueError('first attempt fails')
    return 'ok'

assert flaky() == 'ok'
assert calls['count'] == 2
print('retry calls:', calls['count'])

In [ ]:
# Optional: if src/core/retry.py exists, inspect/import it too.
retry_file = project_root / 'src' / 'core' / 'retry.py'
print('core/retry.py exists:', retry_file.exists())
if retry_file.exists():
    import core.retry as retry_module
    print('core.retry imported:', retry_module)
    print('attributes:', [a for a in dir(retry_module) if 'retry' in a.lower()])